# OTP Login + SQLite + Joke → QRCode


**Workflow:**  
1. Run the SQLite setup.  
2. Run the Register cell → enter username, password, email.  
   - It will show a QR code → scan it with Google Authenticator.  
3. Run the Login cell → enter username, password, and OTP from the app.  
4. If login succeeds → run Joke + QR cells.  


In [ ]:

# Install required packages (only needed in Google Colab)
!pip -q install pyotp qrcode[pil]


In [ ]:

import sqlite3

# Connect to SQLite DB
conn = sqlite3.connect("users.db")
cursor = conn.cursor()

# Create table
cursor.execute('''
CREATE TABLE IF NOT EXISTS users (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    username TEXT UNIQUE,
    password TEXT,
    email TEXT,
    secret TEXT
)
''')
conn.commit()
print("✅ SQLite table ready.")


In [ ]:

import pyotp
import qrcode
from IPython.display import Image, display

def register_user():
    username = input("Enter username: ").strip()
    password = input("Enter password: ").strip()
    email = input("Enter email: ").strip()

    secret = pyotp.random_base32()
    cursor.execute("INSERT OR REPLACE INTO users (username, password, email, secret) VALUES (?, ?, ?, ?)",
                   (username, password, email, secret))
    conn.commit()

    print(f"✅ User '{username}' registered with secret: {secret}")

    # Generate QR for Google Authenticator
    totp = pyotp.TOTP(secret)
    uri = totp.provisioning_uri(name=username, issuer_name="MyApp")
    qr = qrcode.make(uri)
    qr.save("otp_qr.png")
    display(Image(filename="otp_qr.png"))
    print("📱 Scan this QR in Google Authenticator.")

register_user()


In [ ]:

AUTHENTICATED = False
CURRENT_USER = None

def login():
    global AUTHENTICATED, CURRENT_USER

    username = input("Username: ").strip()
    password = input("Password: ").strip()
    otp_code = input("OTP (from Google Authenticator): ").strip()

    cursor.execute("SELECT password, secret FROM users WHERE username=?", (username,))
    row = cursor.fetchone()
    if not row:
        print("❌ User not found")
        return False

    stored_password, secret = row
    if password != stored_password:
        print("❌ Invalid password")
        return False

    totp = pyotp.TOTP(secret)
    if not totp.verify(otp_code):
        print("❌ Invalid OTP")
        return False

    print("✅ Login successful!")
    AUTHENTICATED = True
    CURRENT_USER = username
    return True

login()


In [ ]:

import requests

def get_joke():
    if not AUTHENTICATED:
        raise PermissionError("🔒 Please log in first.")

    url = "https://official-joke-api.appspot.com/random_joke"
    r = requests.get(url, timeout=10)
    data = r.json()
    joke = f"{data['setup']} - {data['punchline']}"
    print("😂 Joke:", joke)
    return joke

joke_text = get_joke()


In [ ]:

def generate_joke_qr(text: str):
    if not AUTHENTICATED:
        raise PermissionError("🔒 Please log in first.")

    img = qrcode.make(text)
    img.save("joke_qr.png")
    display(Image(filename="joke_qr.png"))
    print("✅ QR Code generated for joke.")

generate_joke_qr(joke_text)


In [ ]:
cursor.execute("SELECT * FROM users;")
rows = cursor.fetchall()
for row in rows:
    print(row)